# SR-Site documentary traceability: replication analysis

Package version 2.1.0, 10 October 2026. The notebook runs the executable package workflow and displays its scientific outputs. The principal cohort contains all 55 coverage-elsewhere records outside Project N, with Project N treated separately. PFEP denotes project feature, event or process; IFEP denotes International FEP. Source evidence, proposed search leads and record-review tasks remain distinct. Units, source editions and interpretation rules are documented in the README and protocols.


In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent)
             if (p / "code/reproduce_analysis.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or its notebooks directory.")
DATA = ROOT / "data"
def table(name):
    return pd.read_csv(DATA / name, keep_default_na=False)
def document(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))

def show_figure(name):
    pixels = plt.imread(ROOT / "figures" / name)
    height, width = pixels.shape[:2]
    figure, axis = plt.subplots(figsize=(12, 12 * height / width))
    axis.imshow(pixels)
    axis.set_axis_off()
    figure.tight_layout(pad=0)
    plt.show()
    plt.close(figure)


## Recompute from supplied inputs

Install the pinned dependencies in `requirements.txt` before running this cell. The default command regenerates the documentary audit, auxiliary retrieval, census, 24 ranking configurations, inventory-structure summaries, local record checks and Figures 1, 2 and S1. It uses the supplied review-page accounting. Set `REPORTS` to a directory containing the three official PDF/TXT pairs only when the page frame should be reconstructed; see the external-review protocol for filenames and text extraction.


In [ ]:
REPORTS = None
command = [sys.executable, str(ROOT / "code/reproduce_analysis.py")]
if REPORTS is not None:
    command.extend(["--reports", str(Path(REPORTS).resolve())])
subprocess.run(command, cwd=ROOT, check=True)


## Non-Project N census

The 55 records combine different inventory scopes: 20 have formal heading-only reasons and 35 have general-entry reasons. A record's priority documentary class and the resolvability of any association are separate properties. The alternative relevant-destination criterion is also retained. Domain groupings derive from stored IFEP identifiers and can overlap.


In [ ]:
cohort = document("non_project_n_summary.json")
display(pd.DataFrame({"quantity": ["Non-Project N coverage records", "Formal heading reasons", "General-entry reasons", "Resolvable association", "Unresolved association", "No resolved relevant destination"],
                      "n": [cohort[k] for k in ["cohort_n", "formal_heading_only_n", "formal_too_general_n", "resolvable_association_n", "unresolved_association_n", "no_resolved_relevant_destination_n"]]}))
display(table("non_project_n_inventory_summary.csv"))
display(table("non_project_n_unresolved_records_19.csv"))


## Project N and inventory composition

The formal heading disposition is not the same variable as numeric parenthood. Project N contains 145 coverage records, with 15 numeric parents and 130 numeric leaves. The full-population and non-Project N association statistics describe inventory composition; they do not establish a causal inventory effect.


In [ ]:
display(pd.Series(document("project_n_inventory_structure_summary.json"), name="value"))
display(table("inventory_relevance_association.csv"))
display(table("non_n_heading_coverage_records.csv"))
show_figure("Figure_1_Branching_Documentary_Pathways.png")


## Endpoint recovery under 24 retrieval configurations

Six score specifications, two query variants and two eligibility rules use the same 18 sources and 42 unique imported endpoint pairs. The first table contains the seven manuscript configurations; the second contains all 24. Hit@k measures whether at least one declared endpoint appears among the first k candidates. The endpoint set has incomplete relevance judgments, so these metrics do not estimate precision. The fixed weights and 0.34 lead flag were not optimized on an independent validation set; the ranking comparisons use no cutoff.


In [ ]:
display(table("retrieval_main_table_seven_rows.csv"))
display(table("retrieval_rank_comparison_summary.csv"))


## Source and endpoint ranks with identifier sensitivity

The mask removes audited numeric reference tokens while preserving semantic target names and source titles. The query and deletion tables expose the exact change. Candidate text, vocabulary and inverse document frequencies remain fixed. Every endpoint rank is supplied, including ranks beyond the displayed top 20. Ineligible endpoints have blank ranks. The mask does not create a held-out validation set.


In [ ]:
display(table("retrieval_benchmark_query_texts.csv"))
cases = table("retrieval_rank_comparison_cases.csv")
display(cases.loc[cases.model.eq("archived_weighted_score") & cases.query_variant.eq("unmasked") & cases.eligibility_rule.eq("shared_IFEP_required")])
pairs = table("retrieval_rank_comparison_pairs.csv")
display(pairs.loc[pairs.model.eq("archived_weighted_score") & pairs.query_variant.eq("unmasked") & pairs.eligibility_rule.eq("shared_IFEP_required")])


## Worked paths and engineering routes

The 13 cases represent purposive choices of reference origins, aggregation, ambiguity, applicability and retrieval conditions. The H 1.6.3 and I 165 routes connect their archived relationships to receiving processes and specific engineering checks. The supplied technical evidence gives report editions and source locators. These routes do not calculate a new radionuclide release or establish an effect on licensing decisions.


In [ ]:
display(table("worked_documentary_paths.csv")[["case_id", "source_pfep_id", "path_type", "observed_result", "inference_limit"]])
display(pd.DataFrame(document("technical_route_evidence.json")["cases"]))
display(pd.DataFrame(document("technical_case_handling_rows.json")))
show_figure("Figure_2_Documentary_Pathways.png")


## Review-page frame and selected passages

Page retrieval and passage selection use different units. The supplied frame indexes 256 physical pages and a fixed 185-page assessment frame. Keyword and mandatory-context rules retrieve 86 pages; the 42 selected passages occur on 26 pages. The report-level context table preserves overall assessments and qualifications. Two explicit summary recurrences are retained, and the direction counts do not estimate the balance of the complete reports.


In [ ]:
display(table("external_review_selection_accounting.csv"))
display(table("external_review_context_assessments.csv"))
display(table("external_review_findings_42.csv")[["finding_id", "source", "page", "primary_dimension", "direction", "finding", "scope_note"]])
show_figure("Figure_S1_Review_Page_Frame.png")


## Proposed record contract

The example preserves the three named screening authorities, unspecified source component and separate Buffer receiving context. Relation adoption remains a separate field. The local checker separates record-format failures, application consistency issues and technical review tasks. An archived record can pass the first two checks while still requiring source inspection. The specification describes a proposed interface and local checker, not a deployed service.


In [ ]:
record_check = subprocess.run([sys.executable,
    str(ROOT / "code/record_checks/check_screening_record.py"),
    str(ROOT / "specifications/observed_H_1_6_3.json")],
    cwd=ROOT, check=True, capture_output=True, text=True)
display(json.loads(record_check.stdout))


## Archived 2014 comparison

This supplied comparison concerns an archived release and a different assessment context. Reproducing it requires the official archive and the optional converter documented in the README. It uses the full population and fixed 200-record coverage cohort. It does not reproduce the current preliminary safety analysis report database.


In [ ]:
display(table("database_release_comparison_2014.csv"))


## Optional computational tests

These tests check the declared extraction, retrieval, census and record-consistency rules. Technical scope and substantive screening adequacy require source-based judgment.


In [ ]:
for directory in ["code/extraction", "code/semantic", "code/record_checks"]:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", directory,
                    "-p", "test_*.py"], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT / "code/test_cohort_retrieval_checks.py"),
                "--package", str(ROOT), "--output", str(DATA)], cwd=ROOT, check=True)
